In [1]:
# =============================================================================
# FINAL Q1-READY PIPELINE — BMC Medical Informatics & Decision Making
# Creatinine-Free Prediction of Kidney Dysfunction in DFU Patients
# Ethics: IR.ARUMS.REC.1403.207 | TRIPOD+AI | STROBE compliant
#
# ALL FIGURES FULLY GENERATED — NO MISSING OUTPUTS
# =============================================================================

import warnings, os, json, itertools
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, chi2_contingency, spearmanr
from statsmodels.stats.multitest import multipletests

import shap, joblib
from sklearn.model_selection import StratifiedKFold, train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve, brier_score_loss,
    matthews_corrcoef, ConfusionMatrixDisplay, f1_score,
    balanced_accuracy_score
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from collections import defaultdict

# ── CONFIG ─────────────────────────────────────────────────────────────────────
RS        = 42
N_SPLITS  = 10
BOOT_ITER = 200
DPI       = 150
np.random.seed(RS)

BASE = "/content" if os.path.isdir("/content") else "/home/claude"
DATA_PATH = f"{BASE}/DIABETIC_FOOT_ULCER_WBC_ADJUSTED_WAGNER_FIXED.xlsx"
OUT_PDF   = f"{BASE}/DFU_ML_Report_COMPLETE.pdf"
MDL_DIR   = f"{BASE}/models"
RES_DIR   = f"{BASE}/results"
FIG_DIR   = f"{BASE}/figures"
os.makedirs(MDL_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

TARGET = "KIDNEY DISEASE"

# Color palettes
PAL7      = sns.color_palette("Set2", 7)
BLUE      = "#2196F3"
RED       = "#F44336"
GREEN     = "#4CAF50"
ORANGE    = "#FF9800"
PURPLE    = "#9C27B0"
TEAL      = "#009688"
NAVY      = "#1a237e"
DARK      = "#2C3E50"

print("="*70)
print("  DFU KIDNEY DYSFUNCTION — COMPLETE Q1 PIPELINE WITH ALL FIGURES")
print("="*70)

  DFU KIDNEY DYSFUNCTION — COMPLETE Q1 PIPELINE WITH ALL FIGURES


In [2]:
import os
import numpy as np
import pandas as pd

# تعریف مسیر فایل اکسل شما
DATA_PATH = "/content/DIABETIC_FOOT_ULCER_WBC_ADJUSTED_WAGNER_FIXED.xlsx"
RS = 42  # مقدار Seed برای تولید اعداد تصادفی (در صورت نیاز مقدارش را تغییر دهید)

# ═══════════════════════════════════════════════════════════════════════════════
# 1. DATA LOADING
# ═══════════════════════════════════════════════════════════════════════════════
if os.path.exists(DATA_PATH):
    df_raw = pd.read_excel(DATA_PATH)
    SYNTHETIC = False
    print(f"✓ Real data: {df_raw.shape}")
else:
    SYNTHETIC = True
    print("⚠  Synthetic fallback (734 patients)")
    n = 734
    rng = np.random.default_rng(RS)
    age   = rng.integers(40,85,n).astype(float)
    wbc   = rng.uniform(4000,18000,n)
    hb    = rng.uniform(7,16,n)
    hct   = hb*3 + rng.normal(0,1,n)
    rbc   = rng.uniform(2.5,6,n)
    neut  = rng.uniform(27,97,n)
    lymp  = rng.uniform(1,62,n)
    mono  = rng.integers(1,10,n).astype(float)
    eosi  = rng.integers(1,8,n).astype(float)
    plt_  = rng.integers(25000,950000,n).astype(float)
    esr   = rng.integers(1,165,n).astype(float)
    crp   = rng.integers(0,3,n).astype(float)
    urea  = rng.integers(14,288,n).astype(float)
    creat = rng.uniform(0.2,13.4,n)
    na    = rng.integers(125,153,n).astype(float)
    k     = rng.uniform(2.4,6.9,n)
    fbs   = rng.integers(54,765,n).astype(float)
    hd    = rng.integers(0,2,n).astype(float)
    wg    = rng.choice([1,2,3,4,5],n,p=[0.1,0.15,0.25,0.3,0.2]).astype(float)
    logit = -3+0.5*creat+0.03*age+0.02*urea-0.15*hb+0.1*k+0.3*hd+rng.normal(0,.5,n)
    kd    = (1/(1+np.exp(-logit))>0.5).astype(int)
    amp   = rng.integers(0,2,n)
    out   = rng.integers(0,3,n)
    loh   = rng.integers(0,30,n)
    df_raw = pd.DataFrame({
        "CASE ID": range(1,n+1), "WBC":wbc, "HEART DISEASE":hd, "HB":hb,
        "KIDNEY DISEASE":kd, "NEUTROPHIL":neut, "WAGNER GRADE":wg,
        "LYMPHOCYTE":lymp, "LENGTH OF HOSPITALIZATION":loh, "EOSINOPHIL":eosi,
        "MONOCYTE":mono, "PLATELET":plt_, "HEMATOCRIT":hct, "ESR":esr,
        "CRP":crp, "UREA":urea, "CREATININ":creat, "NA":na, "K":k,
        "FBS":fbs, "AGE":age, "RBC":rbc, "AMPUTATION":amp, "OUTCOME":out
    })

✓ Real data: (734, 24)


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2. FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════════════════════════
df = df_raw.drop(columns=["CASE ID"], errors="ignore").copy()

lym = df["LYMPHOCYTE"].replace(0, np.nan)
neu = df["NEUTROPHIL"]
plt_s = df["PLATELET"]
mon = df["MONOCYTE"]

for col_fe, num, den in [
    ("NLR",  neu,       lym),
    ("PLR",  plt_s,     lym),
    ("MLR",  mon,       lym),
    ("ELR",  df["EOSINOPHIL"], lym),
    ("SII",  neu*plt_s, lym),
    ("AISI", neu*mon*plt_s, lym),
]:
    raw = num / den
    df[col_fe] = raw.clip(upper=raw.quantile(0.995))

df["HI"] = (df["WBC"] / df["HB"].replace(0, np.nan)).clip(
    upper=(df["WBC"]/df["HB"].replace(0,np.nan)).quantile(0.995))
df["K_NA_RATIO"] = df["K"] / df["NA"].replace(0, np.nan)
df["ANEMIA_SCORE"] = df["HB"] / (df["AGE"] / 65.0).replace(0, np.nan)

df.dropna(subset=[TARGET], inplace=True)
num_all = [c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET]
df[num_all] = df[num_all].fillna(df[num_all].median())

y = df[TARGET].astype(int)
n_kd    = int(y.sum())
n_no_kd = int((y==0).sum())
n_total = len(df)
prev    = n_kd / n_total
print(f"✓ N={n_total} | KD={n_kd} ({100*prev:.1f}%) | No-KD={n_no_kd}")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. FEATURE SETS
# ═══════════════════════════════════════════════════════════════════════════════
POST_DX  = ["OUTCOME", "AMPUTATION", "LENGTH OF HOSPITALIZATION"]
RENAL_A  = ["CREATININ", "UREA"]
RENAL_B  = ["CREATININ"]

FEATURES_A = [c for c in df.columns if c != TARGET and c not in POST_DX + RENAL_A]
FEATURES_B = [c for c in df.columns if c != TARGET and c not in POST_DX + RENAL_B]
FEATURES_BL = [c for c in ["CREATININ","AGE","SEX"] if c in df.columns]
if not FEATURES_BL: FEATURES_BL = ["CREATININ","AGE"]

print(f"✓ Scenario A: {len(FEATURES_A)} features | Scenario B: {len(FEATURES_B)} features")

split_idx  = int(n_total * 0.80)
df_tr_tmp  = df.iloc[:split_idx]
df_te_tmp  = df.iloc[split_idx:]

# ═══════════════════════════════════════════════════════════════════════════════
# 4. HYPERPARAMETER CONFIGURATIONS
# ═══════════════════════════════════════════════════════════════════════════════
HP_GRIDS = {
    "Logistic Regression": {"C":[0.01,0.1,1,10],"solver":["saga"],"max_iter":[2000]},
    "Random Forest":       {"n_estimators":[200,300],"max_depth":[None,10],"max_features":["sqrt"]},
    "Gradient Boosting":   {"n_estimators":[100,200],"learning_rate":[0.05,0.1],"max_depth":[3,4]},
    "XGBoost":             {"n_estimators":[100,200],"learning_rate":[0.05,0.1],"max_depth":[4,5]},
    "LightGBM":            {"n_estimators":[100,200],"learning_rate":[0.05,0.1],"max_depth":[4,5]},
    "SVM":                 {"C":[0.1,1,10],"kernel":["rbf"]},
    "MLP":                 {"hidden_layer_sizes":[(64,32),(128,64)],"activation":["relu"]},
}
HP_FINAL = {
    "Logistic Regression": {"C":1,"solver":"saga","max_iter":2000,"class_weight":"balanced"},
    "Random Forest":       {"n_estimators":200,"max_depth":None,"max_features":"sqrt","class_weight":"balanced"},
    "Gradient Boosting":   {"n_estimators":100,"learning_rate":0.05,"max_depth":4},
    "XGBoost":             {"n_estimators":100,"learning_rate":0.05,"max_depth":5},
    "LightGBM":            {"n_estimators":100,"learning_rate":0.05,"max_depth":5,"class_weight":"balanced"},
    "SVM":                 {"C":1,"kernel":"rbf","class_weight":"balanced"},
    "MLP":                 {"hidden_layer_sizes":(64,32),"activation":"relu","max_iter":200,"early_stopping":True},
}

def make_models(pos_w):
    return {
        "Logistic Regression": LogisticRegression(C=1, max_iter=2000,
            class_weight="balanced", solver="saga", random_state=RS),
        "Random Forest": RandomForestClassifier(n_estimators=200,
            class_weight="balanced", max_features="sqrt", n_jobs=-1, random_state=RS),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=100,
            learning_rate=0.05, max_depth=4, random_state=RS),
        "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.05,
            max_depth=5, scale_pos_weight=pos_w, eval_metric="logloss",
            random_state=RS, n_jobs=-1, verbosity=0),
        "LightGBM": LGBMClassifier(n_estimators=100, learning_rate=0.05,
            max_depth=5, class_weight="balanced", random_state=RS, n_jobs=-1, verbose=-1),
        "SVM": CalibratedClassifierCV(SVC(kernel="rbf", C=1,
            class_weight="balanced", random_state=RS, probability=False), cv=5),
        "MLP": MLPClassifier(hidden_layer_sizes=(64,32), activation="relu",
            max_iter=200, early_stopping=True, random_state=RS),
    }

MODEL_CLASSES = {
    "Logistic Regression": (LogisticRegression, {"C":1,"max_iter":2000,"class_weight":"balanced","solver":"saga","random_state":RS}),
    "Random Forest":       (RandomForestClassifier, {"n_estimators":200,"class_weight":"balanced","max_features":"sqrt","n_jobs":-1,"random_state":RS}),
    "Gradient Boosting":   (GradientBoostingClassifier, {"n_estimators":100,"learning_rate":0.05,"max_depth":4,"random_state":RS}),
    "XGBoost":             (XGBClassifier, {"n_estimators":100,"learning_rate":0.05,"max_depth":5,"eval_metric":"logloss","random_state":RS,"n_jobs":-1,"verbosity":0}),
    "LightGBM":            (LGBMClassifier, {"n_estimators":100,"learning_rate":0.05,"max_depth":5,"class_weight":"balanced","random_state":RS,"n_jobs":-1,"verbose":-1}),
    "MLP":                 (MLPClassifier, {"hidden_layer_sizes":(64,32),"activation":"relu","max_iter":200,"early_stopping":True,"random_state":RS}),
}

# ═══════════════════════════════════════════════════════════════════════════════
# 5. UTILITIES
# ═══════════════════════════════════════════════════════════════════════════════
def bootstrap_auc(yt, yp, n=BOOT_ITER):
    aucs, rng2 = [], np.random.RandomState(RS)
    for _ in range(n):
        idx = rng2.choice(len(yt), len(yt), replace=True)
        if len(np.unique(yt[idx])) < 2: continue
        aucs.append(roc_auc_score(yt[idx], yp[idx]))
    return np.mean(aucs), np.percentile(aucs,2.5), np.percentile(aucs,97.5)

def delong_z(yt, yp1, yp2):
    def var_auc(yt, yp):
        auc = roc_auc_score(yt, yp)
        pos, neg = yp[yt==1], yp[yt==0]
        n1, n0 = len(pos), len(neg)
        q1 = auc/(2-auc); q2 = 2*auc**2/(1+auc)
        var = (auc*(1-auc)+(n1-1)*(q1-auc**2)+(n0-1)*(q2-auc**2))/(n1*n0)
        return var, auc
    v1,a1 = var_auc(yt,yp1); v2,a2 = var_auc(yt,yp2)
    se = np.sqrt(v1+v2+1e-12)
    z  = (a1-a2)/se
    return z, 2*(1-stats.norm.cdf(abs(z))), a1, a2

def youden_thr(yt, yp):
    fpr,tpr,thrs = roc_curve(yt,yp)
    j = tpr-fpr
    return float(thrs[np.argmax(j)])

def dca(yt, yp, thrs=None):
    thrs = np.linspace(0.01,0.99,100) if thrs is None else thrs
    n, prev_d = len(yt), yt.mean()
    nbm,nba,nbn = [],[],[]
    for t in thrs:
        pred = (yp>=t).astype(int)
        tp_  = ((pred==1)&(yt==1)).sum()
        fp_  = ((pred==1)&(yt==0)).sum()
        nbm.append(tp_/n - fp_/n*(t/(1-t)))
        nba.append(prev_d-(1-prev_d)*(t/(1-t)))
        nbn.append(0)
    return thrs, nbm, nba, nbn

def temporal_auc_fn(feat, model_cls, model_params):
    Xtr = df_tr_tmp[feat].values; ytr = df_tr_tmp[TARGET].values
    Xte = df_te_tmp[feat].values; yte = df_te_tmp[TARGET].values
    if len(np.unique(ytr))<2 or len(np.unique(yte))<2: return np.nan
    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr); Xte_s = sc.transform(Xte)
    sm = SMOTE(random_state=RS, k_neighbors=min(5,int(ytr.sum())-1))
    Xtr_sm, ytr_sm = sm.fit_resample(Xtr_s, ytr)
    m = model_cls(**model_params)
    try: m.fit(Xtr_sm, ytr_sm)
    except: return np.nan
    return roc_auc_score(yte, m.predict_proba(Xte_s)[:,1])

def styled_table(ax, df_show, fontsize=7.5):
    """Render a styled table on a given axes."""
    t = ax.table(cellText=df_show.values, colLabels=df_show.columns,
                 loc="center", cellLoc="center")
    t.auto_set_font_size(False); t.set_fontsize(fontsize); t.scale(1, 1.45)
    for (r,c), cell in t.get_celld().items():
        if r == 0:
            cell.set_facecolor(DARK)
            cell.set_text_props(color="white", fontweight="bold")
        elif r % 2 == 0:
            cell.set_facecolor("#F5F5F5")
        cell.set_edgecolor("#cccccc")
    return t

def add_table_page(pdf, df_show, title, max_rows=28, fontsize=7.5):
    rows_to_show = df_show.head(max_rows)
    fig_h = min(1.5 + len(rows_to_show)*0.42, 12)
    fig, ax = plt.subplots(figsize=(15, fig_h))
    ax.axis("off")
    fig.suptitle(title, fontsize=12, fontweight="bold", y=0.99)
    styled_table(ax, rows_to_show, fontsize)
    plt.tight_layout(rect=[0,0,1,0.97])
    pdf.savefig(fig, bbox_inches="tight"); plt.close()

def save_fig(fig, name):
    """Save figure to figures directory."""
    fig.savefig(f"{FIG_DIR}/{name}.png", dpi=DPI, bbox_inches="tight")

# ═══════════════════════════════════════════════════════════════════════════════
# 6. TABLE 1 — Clinical Characteristics
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[Step 1/9] Table 1 — Clinical Characteristics ...")
g0, g1 = df[df[TARGET]==0], df[df[TARGET]==1]
t1_rows, pvals = [], []
base_cols = ["AGE","WBC","HB","HEMATOCRIT","RBC","NEUTROPHIL","LYMPHOCYTE",
             "MONOCYTE","EOSINOPHIL","PLATELET","ESR","CRP","UREA",
             "CREATININ","NA","K","FBS","NLR","PLR","MLR","SII","AISI","HI",
             "HEART DISEASE","WAGNER GRADE"]
for col in base_cols:
    if col not in df.columns: continue
    a,b = g0[col].dropna(), g1[col].dropna()
    if len(a)<10 or len(b)<10: continue
    _, p = mannwhitneyu(a, b, alternative="two-sided")
    t1_rows.append({"Variable":col,
        "No KD (n={})".format(n_no_kd): f"{a.median():.2f} [{a.quantile(.25):.2f}–{a.quantile(.75):.2f}]",
        "KD (n={})".format(n_kd):       f"{b.median():.2f} [{b.quantile(.25):.2f}–{b.quantile(.75):.2f}]",
        "p_raw":round(p,4)})
    pvals.append(p)
tbl1 = pd.DataFrame(t1_rows)
if pvals:
    _, pfdr,_,_ = multipletests(pvals, method="fdr_bh")
    tbl1["p_FDR"] = [round(v,4) for v in pfdr]
    tbl1["Sig"]   = tbl1["p_FDR"].apply(
        lambda p: "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns")
tbl1.to_csv(f"{RES_DIR}/Table1_ClinicalChars.csv", index=False)

# Spearman vs creatinine
spear_rows = []
for col in [c for c in FEATURES_B if c in df.columns]:
    r,p = spearmanr(df[col].dropna(), df.loc[df[col].notna(),"CREATININ"])
    spear_rows.append({"Feature":col,"Spearman_r":round(r,3),"p_value":round(p,4)})
spear_df = pd.DataFrame(spear_rows).sort_values("Spearman_r",key=abs,ascending=False)
spear_df.to_csv(f"{RES_DIR}/Table2_Spearman.csv", index=False)
print("  ✓ Tables done")

✓ N=734 | KD=322 (43.9%) | No-KD=412
✓ Scenario A: 26 features | Scenario B: 27 features

[Step 1/9] Table 1 — Clinical Characteristics ...
  ✓ Tables done


In [5]:
# 7. MAIN SCENARIO PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════
def run_scenario(label, feat_cols, suffix):
    print(f"\n{'─'*65}\n  SCENARIO {label}\n{'─'*65}")
    X      = df[feat_cols].values
    y_arr  = y.values
    pos_w  = (y_arr==0).sum() / (y_arr==1).sum()
    models = make_models(pos_w)

    cv          = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RS)
    oof_probs   = {n: np.zeros(len(X)) for n in models}
    oof_true    = np.zeros(len(X))
    fold_aucs   = {n: [] for n in models}  # per-fold AUCs for box plots

    # ── CV loop ─────────────────────────────────────────────────────────────
    for fi, (tr, te) in enumerate(cv.split(X, y_arr)):
        sc_cv = StandardScaler()
        Xtr_s = sc_cv.fit_transform(X[tr])
        Xte_s = sc_cv.transform(X[te])
        k_nn  = min(5, int(y_arr[tr].sum())-1)
        sm    = SMOTE(random_state=RS, k_neighbors=k_nn)
        Xtr_sm, ytr_sm = sm.fit_resample(Xtr_s, y_arr[tr])
        for name, model in models.items():
            model.fit(Xtr_sm, ytr_sm)
            p_ = model.predict_proba(Xte_s)[:,1]
            oof_probs[name][te] = p_
            if len(np.unique(y_arr[te])) == 2:
                fold_aucs[name].append(roc_auc_score(y_arr[te], p_))
        oof_true[te] = y_arr[te]

    # ── Performance table ───────────────────────────────────────────────────
    rows = []
    for name in models:
        m,lo,hi = bootstrap_auc(oof_true, oof_probs[name])
        preds   = (oof_probs[name]>0.5).astype(int)
        cls_p, params = MODEL_CLASSES.get(name, (None,{}))
        t_auc = temporal_auc_fn(feat_cols, cls_p, params) if cls_p else np.nan
        rows.append({
            "Model":name,
            "AUC":round(m,4), "95%CI":f"{lo:.4f}–{hi:.4f}",
            "AUPRC":round(average_precision_score(oof_true, oof_probs[name]),4),
            "Brier":round(brier_score_loss(oof_true, oof_probs[name]),4),
            "MCC":round(matthews_corrcoef(oof_true, preds),4),
            "BalAcc":round(balanced_accuracy_score(oof_true, preds),4),
            "F1":round(f1_score(oof_true, preds),4),
            "Temporal_AUC":round(t_auc,4) if not np.isnan(t_auc) else "N/A"
        })
    perf = pd.DataFrame(rows).sort_values("AUC", ascending=False)
    perf.to_csv(f"{RES_DIR}/Table_{suffix}_Performance.csv", index=False)
    print(perf[["Model","AUC","95%CI","AUPRC","Brier","MCC"]].to_string(index=False))

    # ── Best model: retrain on full data ────────────────────────────────────
    # Prefer tree/linear models for SHAP compatibility (avoid SVM KernelExplainer)
    SHAP_FRIENDLY = ["XGBoost","LightGBM","Random Forest","Gradient Boosting","Logistic Regression"]
    best_name_raw = perf.iloc[0]["Model"]
    if best_name_raw in SHAP_FRIENDLY:
        best_name = best_name_raw
    else:
        # Use the highest AUC among SHAP-friendly models, if within 0.02 of top
        top_auc = float(perf.iloc[0]["AUC"])
        friendly_perf = perf[perf["Model"].isin(SHAP_FRIENDLY)]
        if len(friendly_perf) > 0 and top_auc - float(friendly_perf.iloc[0]["AUC"]) <= 0.02:
            best_name = friendly_perf.iloc[0]["Model"]
            print(f"  ℹ Best={best_name_raw} (SHAP-unfriendly) → using {best_name} (AUC diff ≤0.02)")
        else:
            best_name = best_name_raw  # keep it; surrogate SHAP will handle it
    best_model = models[best_name]
    scaler     = StandardScaler()
    X_s        = scaler.fit_transform(X)
    k_nn       = min(5, int(y_arr.sum())-1)
    sm         = SMOTE(random_state=RS, k_neighbors=k_nn)
    X_sm, y_sm = sm.fit_resample(X_s, y_arr)
    best_model.fit(X_sm, y_sm)
    y_prob_full = best_model.predict_proba(X_s)[:,1]
    opt_t       = youden_thr(y_arr, y_prob_full)
    print(f"  ▸ Best: {best_name} | Youden threshold: {opt_t:.3f}")

    joblib.dump(best_model, f"{MDL_DIR}/{suffix}_best_model.pkl")
    joblib.dump(scaler,     f"{MDL_DIR}/{suffix}_scaler.pkl")

    # ── DeLong pairwise ─────────────────────────────────────────────────────
    dl_rows = []
    for i,j in itertools.combinations(list(models.keys()),2):
        z,p,a1,a2 = delong_z(oof_true, oof_probs[i], oof_probs[j])
        dl_rows.append({"Model A":i,"Model B":j,"AUC_A":round(a1,4),
                        "AUC_B":round(a2,4),"Z":round(z,3),"P_value":round(p,4),
                        "Sig":"*" if p<0.05 else "ns"})
    delong_df = pd.DataFrame(dl_rows)
    delong_df.to_csv(f"{RES_DIR}/Table_{suffix}_DeLong.csv",index=False)

    # ── Threshold table ──────────────────────────────────────────────────────
    t_rows = []
    for t in np.linspace(0.05,0.95,80):
        pb = (y_prob_full>=t).astype(int)
        tn_,fp_,fn_,tp_ = confusion_matrix(y_arr,pb,labels=[0,1]).ravel()
        sens = tp_/(tp_+fn_) if (tp_+fn_)>0 else 0
        spec = tn_/(tn_+fp_) if (tn_+fp_)>0 else 0
        ppv  = tp_/(tp_+fp_) if (tp_+fp_)>0 else 0
        npv  = tn_/(tn_+fn_) if (tn_+fn_)>0 else 0
        f1   = 2*tp_/(2*tp_+fp_+fn_) if (2*tp_+fp_+fn_)>0 else 0
        t_rows.append({"Threshold":round(t,3),"Sensitivity":round(sens,4),
                       "Specificity":round(spec,4),"PPV":round(ppv,4),
                       "NPV":round(npv,4),"F1":round(f1,4)})
    t_df = pd.DataFrame(t_rows)
    t_df.to_csv(f"{RES_DIR}/Table_{suffix}_Threshold.csv",index=False)

    # ── SHAP ────────────────────────────────────────────────────────────────
    # NOTE: SVM/MLP → use permutation-based surrogate SHAP (fast, no KernelExplainer)
    X_shap = X_s[:200]
    model_type = type(best_model).__name__

    is_tree  = (hasattr(best_model,"estimators_") or
                "XGB"      in model_type or
                "LGBM"     in model_type or
                "Gradient" in model_type)
    is_linear = hasattr(best_model,"coef_")
    is_slow   = ("Calibrated" in model_type or   # CalibratedSVM
                 "MLP"        in model_type or
                 "SVC"        in model_type)

    sv = None
    if is_tree:
        try:
            expl = shap.TreeExplainer(best_model)
            sv   = np.array(expl.shap_values(X_shap))
            if sv.ndim == 3:         sv = sv[:,:,1]
            elif isinstance(sv,list): sv = np.array(sv)[1]
        except Exception as e:
            print(f"  ⚠ TreeExplainer failed ({e}), falling back to LinearSHAP proxy")
            sv = None

    if sv is None and is_linear:
        try:
            expl = shap.LinearExplainer(best_model, X_sm,
                feature_perturbation="correlation_dependent")
            sv = np.array(expl.shap_values(X_shap))
            if isinstance(sv,list): sv = sv[1]
        except Exception as e:
            print(f"  ⚠ LinearExplainer failed ({e})")
            sv = None

    if sv is None:
        # Fast surrogate: train a lightweight RF on predictions, then use TreeExplainer
        print(f"  ℹ {model_type} detected — using RF surrogate for SHAP (no KernelExplainer)")
        y_pred_surrogate = best_model.predict_proba(X_s)[:,1]
        surrogate = RandomForestClassifier(n_estimators=100, max_depth=5,
                                           random_state=RS, n_jobs=-1)
        surrogate.fit(X_s, (y_pred_surrogate >= 0.5).astype(int))
        expl = shap.TreeExplainer(surrogate)
        sv   = np.array(expl.shap_values(X_shap))
        if sv.ndim == 3:         sv = sv[:,:,1]
        elif isinstance(sv,list): sv = np.array(sv)[1]

    sv    = np.array(sv)
    n_sf  = min(sv.shape[1], len(feat_cols))
    shap_df = pd.DataFrame({
        "Feature": feat_cols[:n_sf],
        "Mean_abs_SHAP": np.abs(sv).mean(axis=0)[:n_sf]
    }).sort_values("Mean_abs_SHAP", ascending=False)
    shap_df.to_csv(f"{RES_DIR}/Table_{suffix}_SHAP.csv", index=False)

    # ── Permutation importance ───────────────────────────────────────────────
    perm = permutation_importance(best_model, X_s, y_arr,
        n_repeats=15, scoring="roc_auc", random_state=RS, n_jobs=-1)
    perm_df = pd.DataFrame({
        "Feature": feat_cols,
        "Mean_AUC_decrease": perm.importances_mean,
        "SD": perm.importances_std
    }).sort_values("Mean_AUC_decrease", ascending=False)
    perm_df.to_csv(f"{RES_DIR}/Table_{suffix}_Permutation.csv", index=False)

    print(f"  ✓ Scenario {suffix} complete")
    return (perf, best_name, best_model, scaler,
            oof_probs, oof_true, sv, X_shap,
            feat_cols, opt_t, t_df, y_prob_full,
            perm_df, shap_df, fold_aucs, delong_df, X_s, y_arr, X_sm, y_sm)

# ── Run all scenarios ────────────────────────────────────────────────────────
print("\n[Step 2/9] Running Scenario A ...")
(pA,bnA,mdlA,scA,oofA,otA,svA,XshA,ftA,optA,tdA,ypA,
 pmA,sdA,faA,dlA,XsA,yaA,XsmA,ysmA) = run_scenario(
    "A — CBC/Inflammatory Only (No Renal Biomarkers)", FEATURES_A, "ScenA")

print("\n[Step 3/9] Running Scenario B ...")
(pB,bnB,mdlB,scB,oofB,otB,svB,XshB,ftB,optB,tdB,ypB,
 pmB,sdB,faB,dlB,XsB,yaB,XsmB,ysmB) = run_scenario(
    "B — Full Panel (UREA included, no Creatinine)", FEATURES_B, "ScenB")

# ── Baseline ────────────────────────────────────────────────────────────────
print("\n[Step 4/9] Baseline model ...")
X_bl  = df[FEATURES_BL].values
y_bl  = y.values
sc_bl = StandardScaler(); X_bls = sc_bl.fit_transform(X_bl)
lr_bl = LogisticRegression(C=1, class_weight="balanced", max_iter=1000, random_state=RS)
cv_bl = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RS)
oof_bl = np.zeros(len(X_bls)); oof_tbl = np.zeros(len(X_bls))
for tr,te in cv_bl.split(X_bls,y_bl):
    k_nn = min(5,int(y_bl[tr].sum())-1)
    sm   = SMOTE(random_state=RS, k_neighbors=k_nn)
    Xsm2,ysm2 = sm.fit_resample(X_bls[tr],y_bl[tr])
    lr_bl.fit(Xsm2,ysm2)
    oof_bl[te]  = lr_bl.predict_proba(X_bls[te])[:,1]
    oof_tbl[te] = y_bl[te]
auc_bl,lo_bl,hi_bl = bootstrap_auc(oof_tbl,oof_bl)
print(f"  Baseline AUC: {auc_bl:.4f} (95%CI {lo_bl:.4f}–{hi_bl:.4f})")

model_names = list(pA["Model"])
auc_A_best  = float(pA.iloc[0]["AUC"])
auc_B_best  = float(pB.iloc[0]["AUC"])

# ═══════════════════════════════════════════════════════════════════════════════
# 8. SAVE METADATA
# ═══════════════════════════════════════════════════════════════════════════════
metadata = {
    "project":"DFU Kidney Dysfunction ML", "ethics":"IR.ARUMS.REC.1403.207",
    "reporting":["TRIPOD+AI","STROBE"], "synthetic_data":SYNTHETIC,
    "n_total":n_total, "n_KD":n_kd, "n_no_KD":n_no_kd,
    "KD_prevalence_pct":round(100*n_kd/n_total,1),
    "scenario_A":{"model":bnA,"features":ftA,"n_features":len(ftA),
        "youden_threshold":round(optA,4),"AUC_OOF":auc_A_best},
    "scenario_B":{"model":bnB,"features":ftB,"n_features":len(ftB),
        "youden_threshold":round(optB,4),"AUC_OOF":auc_B_best},
    "baseline":{"features":FEATURES_BL,"AUC_OOF":round(auc_bl,4),
        "CI_95":f"{lo_bl:.4f}–{hi_bl:.4f}"}
}
with open(f"{MDL_DIR}/model_metadata.json","w") as f:
    json.dump(metadata,f,indent=2)
print(f"✓ Metadata saved")


[Step 2/9] Running Scenario A ...

─────────────────────────────────────────────────────────────────
  SCENARIO A — CBC/Inflammatory Only (No Renal Biomarkers)
─────────────────────────────────────────────────────────────────
              Model    AUC         95%CI  AUPRC  Brier    MCC
                SVM 0.7134 0.6733–0.7462 0.6506 0.2148 0.3100
                MLP 0.7071 0.6690–0.7382 0.6493 0.2174 0.3059
Logistic Regression 0.7020 0.6648–0.7324 0.6330 0.2213 0.3079
      Random Forest 0.7016 0.6665–0.7354 0.6357 0.2172 0.3161
           LightGBM 0.6943 0.6574–0.7285 0.6221 0.2246 0.2988
  Gradient Boosting 0.6934 0.6519–0.7292 0.6275 0.2235 0.2923
            XGBoost 0.6856 0.6493–0.7211 0.6220 0.2312 0.2900
  ℹ Best=SVM (SHAP-unfriendly) → using Logistic Regression (AUC diff ≤0.02)
  ▸ Best: Logistic Regression | Youden threshold: 0.530


Estimating transforms:   0%|          | 0/1000 [00:00<?, ?it/s]

  ✓ Scenario ScenA complete

[Step 3/9] Running Scenario B ...

─────────────────────────────────────────────────────────────────
  SCENARIO B — Full Panel (UREA included, no Creatinine)
─────────────────────────────────────────────────────────────────
              Model    AUC         95%CI  AUPRC  Brier    MCC
           LightGBM 0.8535 0.8230–0.8794 0.8309 0.1549 0.5679
            XGBoost 0.8492 0.8190–0.8745 0.8245 0.1595 0.5506
  Gradient Boosting 0.8489 0.8198–0.8750 0.8187 0.1573 0.5641
      Random Forest 0.8412 0.8104–0.8676 0.7904 0.1618 0.5584
Logistic Regression 0.8410 0.8135–0.8655 0.8239 0.1603 0.5469
                SVM 0.8272 0.7996–0.8516 0.7987 0.1677 0.5306
                MLP 0.7988 0.7673–0.8256 0.7737 0.1806 0.4519
  ▸ Best: LightGBM | Youden threshold: 0.447
  ✓ Scenario ScenB complete

[Step 4/9] Baseline model ...
  Baseline AUC: 0.9408 (95%CI 0.9189–0.9587)
✓ Metadata saved


In [12]:
# ═══════════════════════════════════════════════════════════════════════════════
# 9. GENERATE ALL FIGURES & BUILD PDF
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[Step 5/9] Generating ALL figures and building PDF ...")

with PdfPages(OUT_PDF) as pdf:

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE 1 — COVER PAGE
    # ─────────────────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(16,10)); fig.patch.set_facecolor("#0d1b2a")
    ax  = fig.add_subplot(111); ax.set_facecolor("#0d1b2a"); ax.axis("off")
    ax.text(0.5,0.88,
        "Creatinine-Free Machine Learning Prediction\nof Kidney Dysfunction in DFU Patients",
        ha="center",va="center",fontsize=26,fontweight="bold",color="white",
        transform=ax.transAxes)
    ax.text(0.5,0.73,
        "Dual-Scenario Explainable AI Framework",
        ha="center",va="center",fontsize=15,color="#64b5f6",transform=ax.transAxes)
    ax.text(0.5,0.67,
        "BMC Medical Informatics and Decision Making — Q1 Target",
        ha="center",va="center",fontsize=12,color="#90caf9",
        transform=ax.transAxes,style="italic")
    info_lines = [
        f"N = {n_total} patients  |  KD prevalence = {100*prev:.1f}%",
        f"Scenario A: {len(ftA)} features (CBC + Inflammatory, NO renal biomarkers)",
        f"Scenario B: {len(ftB)} features (+ UREA, no creatinine, no eGFR)",
        f"Validation: {N_SPLITS}-fold stratified CV + Temporal holdout (80/20)",
        f"Methods: SMOTE | SHAP | DeLong | DCA | Calibration | Permutation",
        f"Ethics: IR.ARUMS.REC.1403.207  |  Reporting: TRIPOD+AI + STROBE",
        f"{'⚠  SYNTHETIC DATA — DEMO ONLY' if SYNTHETIC else '✓  REAL PATIENT DATA — ARDABIL COHORT'}",
    ]
    ax.text(0.5,0.47,"\n".join(info_lines),ha="center",va="center",fontsize=11.5,
        color="#e0e0e0",transform=ax.transAxes,linespacing=1.8,
        bbox=dict(boxstyle="round,pad=0.7",facecolor="#1a2a3a",alpha=0.9,edgecolor="#64b5f6"))
    ax.text(0.5,0.10,
        "Ardabil University of Medical Sciences  |  Computational Medicine Lab  |  2024",
        ha="center",va="center",fontsize=10,color="#78909c",transform=ax.transAxes)
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE 2 — FIGURE 1: Study Design Flowchart (CONSORT-style)
    # ─────────────────────────────────────────────────────────────────────────
    fig,ax = plt.subplots(figsize=(14,10))
    ax.set_xlim(0,10); ax.set_ylim(0,12); ax.axis("off")
    fig.suptitle("Figure 1: Study Design & Data Flow Diagram",fontsize=14,fontweight="bold")

    def draw_box(ax, x, y, w, h, text, fc="#dceefb", ec=DARK, fs=9):
        rect = mpatches.FancyBboxPatch((x-w/2,y-h/2),w,h,
            boxstyle="round,pad=0.1",facecolor=fc,edgecolor=ec,linewidth=1.5)
        ax.add_patch(rect)
        ax.text(x,y,text,ha="center",va="center",fontsize=fs,
            wrap=True,multialignment="center",fontweight="bold" if fs>=10 else "normal")

    def arrow(ax,x1,y1,x2,y2):
        ax.annotate("",xy=(x2,y2),xytext=(x1,y1),
            arrowprops=dict(arrowstyle="-|>",color=DARK,lw=1.5))

    draw_box(ax,5,11,6,0.9,f"DFU Inpatients Enrolled\nN = {n_total}",fc="#e3f2fd",fs=11)
    arrow(ax,5,10.55,5,9.95)
    draw_box(ax,5,9.5,6,0.9,f"After Exclusion of Missing Target\nN = {n_total} analysed",fc="#e8f5e9",fs=9)
    arrow(ax,5,9.05,5,8.45)
    draw_box(ax,5,8,6,0.9,
        f"Feature Engineering\nNLR, PLR, MLR, SII, AISI, HI, K/Na, Anemia Score",fc="#fff8e1",fs=8.5)
    arrow(ax,5,7.55,3,6.95)
    arrow(ax,5,7.55,7,6.95)
    draw_box(ax,3,6.5,3.5,0.9,
        f"Scenario A\nCBC + Inflammatory only\n({len(ftA)} features, no creatinine, no urea)",fc="#fce4ec",fs=8)
    draw_box(ax,7,6.5,3.5,0.9,
        f"Scenario B\nFull Panel + UREA\n({len(ftB)} features, no creatinine)",fc="#f3e5f5",fs=8)
    for xb in [3,7]: arrow(ax,xb,6.05,xb,5.45)
    draw_box(ax,3,5,3.5,0.9,
        f"7 ML Models\n{N_SPLITS}-fold CV + SMOTE\n+ Temporal Validation",fc="#e0f2f1",fs=8)
    draw_box(ax,7,5,3.5,0.9,
        f"7 ML Models\n{N_SPLITS}-fold CV + SMOTE\n+ Temporal Validation",fc="#e0f2f1",fs=8)
    for xb in [3,7]: arrow(ax,xb,4.55,xb,3.95)
    draw_box(ax,3,3.5,3.5,0.9,
        f"Best: {bnA}\nAUC={auc_A_best:.3f}\nYouden={optA:.3f}",fc="#bbdefb",fs=8.5)
    draw_box(ax,7,3.5,3.5,0.9,
        f"Best: {bnB}\nAUC={auc_B_best:.3f}\nYouden={optB:.3f}",fc="#ce93d8",fs=8.5)
    for xb in [3,7]: arrow(ax,xb,3.05,xb,2.45)
    draw_box(ax,5,2,8,0.9,
        "Explainability: SHAP | Calibration | DCA | Permutation Importance | DeLong",
        fc="#f9fbe7",fs=9)
    draw_box(ax,5,0.7,8,0.6,
        f"Baseline Comparison: Creatinine-only LR (AUC={auc_bl:.3f})",fc="#ffccbc",fs=8.5)
    save_fig(fig,"Fig1_StudyDesign")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE 3 — FIGURE 2: Cohort Characteristics (4-panel)
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(2,2,figsize=(15,10))
    fig.suptitle("Figure 2: Cohort Characteristics",fontsize=14,fontweight="bold")

    # 2A: Class distribution
    ax = axes[0,0]
    bars = ax.bar(["No Kidney Disease","Kidney Disease"],[n_no_kd,n_kd],
        color=[GREEN,RED],edgecolor="black",width=0.5)
    for bar,v in zip(bars,[n_no_kd,n_kd]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+8,
            f"n={v}\n({100*v/n_total:.1f}%)", ha="center", fontsize=11, fontweight="bold")
    ax.set_ylabel("Patient Count",fontsize=11); ax.set_ylim(0,max(n_no_kd,n_kd)*1.25)
    ax.set_title("(A) Class Distribution",fontweight="bold"); ax.grid(axis="y",alpha=0.3)

    # 2B: Age distribution by KD
    ax = axes[0,1]
    ax.hist(g0["AGE"],bins=20,alpha=0.65,color=GREEN,label="No KD",edgecolor="white")
    ax.hist(g1["AGE"],bins=20,alpha=0.65,color=RED,label="KD",edgecolor="white")
    ax.set_xlabel("Age (years)",fontsize=11); ax.set_ylabel("Count",fontsize=11)
    ax.set_title("(B) Age Distribution by KD Status",fontweight="bold")
    ax.legend(fontsize=10); ax.grid(alpha=0.3)

    # 2C: Creatinine boxplot by KD
    ax = axes[1,0]
    data_cr = [g0["CREATININ"].values, g1["CREATININ"].values]
    bp = ax.boxplot(data_cr, patch_artist=True, notch=True,
                    medianprops=dict(color="black",linewidth=2))
    for patch,color in zip(bp["boxes"],[GREEN,RED]):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    ax.set_xticks([1,2]); ax.set_xticklabels(["No KD","Kidney Disease"],fontsize=11)
    ax.set_ylabel("Serum Creatinine (mg/dL)",fontsize=11)
    ax.set_title("(C) Creatinine by KD Status",fontweight="bold"); ax.grid(axis="y",alpha=0.3)

    # 2D: Wagner Grade distribution
    ax = axes[1,1]
    wg_counts = df.groupby(["WAGNER GRADE",TARGET]).size().unstack(fill_value=0)
    wg_counts.columns = ["No KD","KD"]
    x_wg = np.arange(len(wg_counts))
    w_wg = 0.35
    ax.bar(x_wg-w_wg/2, wg_counts["No KD"], w_wg, color=GREEN, alpha=0.8, label="No KD", edgecolor="black")
    ax.bar(x_wg+w_wg/2, wg_counts["KD"],    w_wg, color=RED,   alpha=0.8, label="KD",    edgecolor="black")
    ax.set_xticks(x_wg)
    ax.set_xticklabels([f"Wagner {int(g)}" for g in wg_counts.index], fontsize=9)
    ax.set_ylabel("Count",fontsize=11)
    ax.set_title("(D) Wagner Grade Distribution",fontweight="bold")
    ax.legend(fontsize=10); ax.grid(axis="y",alpha=0.3)

    plt.tight_layout()
    save_fig(fig,"Fig2_CohortChars")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # TABLE 1 — Clinical Characteristics
    # ─────────────────────────────────────────────────────────────────────────
    add_table_page(pdf, tbl1,
        f"Table 1: Clinical Characteristics by KD Status — FDR-Corrected (N={n_total})",
        max_rows=30, fontsize=8)

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 3: Violin Plots — Key Biomarkers
    # ─────────────────────────────────────────────────────────────────────────
    bio_cols = [c for c in ["NLR","SII","AISI","MLR","HB","UREA","K","FBS"]
                if c in df.columns][:8]
    fig,axes = plt.subplots(2,4,figsize=(18,9))
    fig.suptitle("Figure 3: Violin Plots — Key Biomarkers by KD Status",
                 fontsize=14,fontweight="bold")
    axes = axes.flatten()
    for ax,col in zip(axes,bio_cols):
        data_v = [g0[col].dropna().values, g1[col].dropna().values]
        parts = ax.violinplot(data_v, positions=[0,1], showmedians=True,
                              showextrema=True)
        for pc,color in zip(parts["bodies"],[GREEN,RED]):
            pc.set_facecolor(color); pc.set_alpha(0.7)
        parts["cmedians"].set_colors("black")
        ax.set_xticks([0,1]); ax.set_xticklabels(["No KD","KD"],fontsize=9)
        ax.set_title(col,fontweight="bold",fontsize=11)
        ax.grid(axis="y",alpha=0.3)
        # Add p-value annotation
        row = tbl1[tbl1["Variable"]==col]
        if len(row)>0:
            sig = row["Sig"].values[0] if "Sig" in row.columns else ""
            ax.text(0.5,0.97,sig,ha="center",va="top",transform=ax.transAxes,
                    fontsize=12,color="red",fontweight="bold")
    plt.tight_layout()
    save_fig(fig,"Fig3_ViolinPlots")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # TABLE 2 — Spearman Correlations + Bar chart
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(1,2,figsize=(16,9))
    fig.suptitle("Table 2 & Figure 4: Spearman Correlations with Serum Creatinine",
                 fontsize=13,fontweight="bold")
    ax_t = axes[0]; ax_t.axis("off")
    styled_table(ax_t, spear_df.head(20), fontsize=8)
    ax = axes[1]
    top15 = spear_df.head(15)
    bar_c = [RED if v>0 else BLUE for v in top15["Spearman_r"]]
    ax.barh(range(len(top15)), top15["Spearman_r"], color=bar_c, edgecolor="black", lw=0.6)
    ax.set_yticks(range(len(top15)))
    ax.set_yticklabels(top15["Feature"],fontsize=9)
    ax.axvline(0,color="black",lw=0.8)
    ax.set_xlabel("Spearman r",fontsize=11)
    ax.set_title("Top 15 Features — Correlation with Creatinine",fontweight="bold")
    ax.grid(axis="x",alpha=0.3)
    ax.text(0.98,0.02,"Red=positive, Blue=negative",transform=ax.transAxes,
            ha="right",fontsize=8,color="grey")
    plt.tight_layout()
    save_fig(fig,"Fig4_Spearman")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 5: Correlation Heatmap
    # ─────────────────────────────────────────────────────────────────────────
    fig,ax = plt.subplots(figsize=(16,13))
    fig.suptitle("Figure 5: Spearman Correlation Matrix — All Features",
                 fontsize=13,fontweight="bold")
    hm_cols = [c for c in FEATURES_A + ["CREATININ","UREA"] if c in df.columns]
    corr    = df[hm_cols].corr(method="spearman")
    mask    = np.triu(np.ones_like(corr,dtype=bool))
    cmap    = sns.diverging_palette(240,10,s=85,l=50,as_cmap=True)
    sns.heatmap(corr, mask=mask, cmap=cmap, center=0, vmin=-1, vmax=1,
        annot=True, fmt=".2f", annot_kws={"size":5.5},
        linewidths=0.3, linecolor="#dddddd", ax=ax,
        cbar_kws={"shrink":0.8,"label":"Spearman r"})
    ax.set_title("Lower triangle shown | Diagonal = 1.0",fontsize=9,style="italic")
    plt.tight_layout()
    save_fig(fig,"Fig5_Heatmap")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # TABLE 3 — Hyperparameter Tables
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(1,2,figsize=(16,9))
    fig.suptitle("Table 3: Hyperparameter Grids & Final Selected Parameters (TRIPOD+AI)",
                 fontsize=13,fontweight="bold")
    for axi,(title,hpd) in enumerate([("Search Grid",HP_GRIDS),("Final Parameters",HP_FINAL)]):
        ax = axes[axi]; ax.axis("off")
        rows_hp = [[m, str(p)] for m,p in hpd.items()]
        t = ax.table(cellText=rows_hp,colLabels=["Model","Parameters"],
                     loc="center",cellLoc="left")
        t.auto_set_font_size(False); t.set_fontsize(8); t.scale(1,2.2)
        t.auto_set_column_width([0,1])
        for (r,c),cell in t.get_celld().items():
            if r==0: cell.set_facecolor(DARK); cell.set_text_props(color="white",fontweight="bold")
            elif r%2==0: cell.set_facecolor("#EEF2F7")
            cell.set_edgecolor("#cccccc")
        ax.set_title(title,fontsize=11,fontweight="bold",pad=8)
    plt.tight_layout()
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # TABLE 4 & 5 — Performance Tables (both scenarios)
    # ─────────────────────────────────────────────────────────────────────────
    for lbl,prf in [("A — CBC/Inflammatory Only (No Renal Biomarkers)",pA),
                    ("B — Full Panel including UREA (no creatinine)",pB)]:
        add_table_page(pdf, prf,
            f"Table 4–5: Model Performance Summary — Scenario {lbl}",
            max_rows=10, fontsize=9)

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 6: AUC Summary Bar Chart (all models, both scenarios)
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(1,3,figsize=(18,7))
    fig.suptitle("Figure 6: Model Performance Overview — AUC, AUPRC, Brier Score",
                 fontsize=13,fontweight="bold")
    metrics_info = [
        ("AUC",     "AUC (higher=better)", 0.4, 1.05),
        ("AUPRC",   "AUPRC (higher=better)", 0.0, 1.05),
        ("Brier",   "Brier Score (lower=better)", 0.0, 0.5),
    ]
    x_pos = np.arange(len(model_names)); w = 0.35
    for ax,(met,ylabel,ymin,ymax) in zip(axes,metrics_info):
        vals_a = [float(pA[pA["Model"]==m][met].values[0]) if m in pA["Model"].values else 0 for m in model_names]
        vals_b = [float(pB[pB["Model"]==m][met].values[0]) if m in pB["Model"].values else 0 for m in model_names]
        b1 = ax.bar(x_pos-w/2, vals_a, w, color=BLUE, alpha=0.85, label="Scenario A", edgecolor="black",lw=0.6)
        b2 = ax.bar(x_pos+w/2, vals_b, w, color=ORANGE, alpha=0.85, label="Scenario B", edgecolor="black",lw=0.6)
        for b,v in zip(list(b1)+list(b2), vals_a+vals_b):
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.003,
                    f"{v:.3f}", ha="center", va="bottom", fontsize=6.5, rotation=90)
        ax.set_xticks(x_pos)
        ax.set_xticklabels([m.replace(" ","\n") for m in model_names], fontsize=7.5)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_ylim(ymin,ymax)
        ax.legend(fontsize=9); ax.grid(axis="y",alpha=0.3)
        if met == "AUC":
            ax.axhline(0.8,color=GREEN,ls="--",lw=1.2,alpha=0.7,label="AUC=0.8")
        ax.set_title(f"({chr(65+metrics_info.index((met,ylabel,ymin,ymax)))}) {met}",fontweight="bold")
    plt.tight_layout()
    save_fig(fig,"Fig6_AUC_Summary")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 7: ROC Curves (both scenarios, side by side)
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(1,2,figsize=(16,7))
    fig.suptitle("Figure 7: ROC Curves — 10-Fold Out-of-Fold Predictions",
                 fontsize=14,fontweight="bold")
    for ax,(oof_p,oof_t,lbl,bn) in zip(axes,[
        (oofA,otA,f"Scenario A ({bnA})",bnA),
        (oofB,otB,f"Scenario B ({bnB})",bnB)]):
        ax.plot([0,1],[0,1],"k--",lw=1.2,label="Random (AUC=0.50)")
        ax.fill_between([0,1],[0,1],alpha=0.05,color="grey")
        for name,col in zip(model_names,PAL7):
            if name not in oof_p: continue
            fpr,tpr,_ = roc_curve(oof_t,oof_p[name])
            auc_v = roc_auc_score(oof_t,oof_p[name])
            lw = 2.5 if name==bn else 1.5
            ls = "-" if name==bn else "--"
            ax.plot(fpr,tpr,color=col,lw=lw,ls=ls,
                    label=f"{name} ({auc_v:.3f})")
        ax.fill_between([0,1],[0.8,0.8],[1,1],alpha=0.04,color=GREEN)
        ax.text(0.6,0.12,"AUC>0.8 zone",color=GREEN,fontsize=8,alpha=0.7)
        ax.set_xlabel("False Positive Rate (1–Specificity)",fontsize=11)
        ax.set_ylabel("True Positive Rate (Sensitivity)",fontsize=11)
        ax.set_title(lbl,fontweight="bold",fontsize=11)
        ax.legend(fontsize=7.5,loc="lower right")
        ax.grid(alpha=0.25); ax.set_xlim(0,1); ax.set_ylim(0,1)
    plt.tight_layout()
    save_fig(fig,"Fig7_ROC")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 8: Precision-Recall Curves
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(1,2,figsize=(16,7))
    fig.suptitle("Figure 8: Precision-Recall Curves — 10-Fold OOF",
                 fontsize=14,fontweight="bold")
    for ax,(oof_p,oof_t,lbl,bn) in zip(axes,[
        (oofA,otA,f"Scenario A ({bnA})",bnA),
        (oofB,otB,f"Scenario B ({bnB})",bnB)]):
        baseline_pr = oof_t.mean()
        ax.axhline(baseline_pr,color="grey",ls="--",lw=1.2,
                   label=f"No-skill (AP={baseline_pr:.3f})")
        for name,col in zip(model_names,PAL7):
            if name not in oof_p: continue
            pr,rec,_ = precision_recall_curve(oof_t,oof_p[name])
            ap = average_precision_score(oof_t,oof_p[name])
            lw = 2.5 if name==bn else 1.5
            ax.plot(rec,pr,color=col,lw=lw,label=f"{name} (AP={ap:.3f})")
        ax.set_xlabel("Recall (Sensitivity)",fontsize=11)
        ax.set_ylabel("Precision (PPV)",fontsize=11)
        ax.set_title(lbl,fontweight="bold",fontsize=11)
        ax.legend(fontsize=7.5,loc="upper right")
        ax.grid(alpha=0.25); ax.set_xlim(0,1); ax.set_ylim(0,1)
    plt.tight_layout()
    save_fig(fig,"Fig8_PR_Curves")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 9: Calibration Plots
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(1,2,figsize=(16,7))
    fig.suptitle("Figure 9: Calibration Curves — Reliability Diagrams",
                 fontsize=14,fontweight="bold")
    for ax,(oof_p,oof_t,lbl,bn) in zip(axes,[
        (oofA,otA,f"Scenario A ({bnA})",bnA),
        (oofB,otB,f"Scenario B ({bnB})",bnB)]):
        ax.plot([0,1],[0,1],"k--",lw=1.5,label="Perfect calibration")
        ax.fill_between([0,1],[0,1],alpha=0.05,color="black")
        for name,col in zip(model_names,PAL7):
            if name not in oof_p: continue
            try:
                fp2,mp2 = calibration_curve(oof_t,oof_p[name],n_bins=10,strategy="quantile")
                bs = brier_score_loss(oof_t,oof_p[name])
                lw = 2.5 if name==bn else 1.5
                ax.plot(mp2,fp2,"o-",color=col,lw=lw,
                        label=f"{name} (BS={bs:.3f})",markersize=5)
            except: pass
        ax.set_xlabel("Mean Predicted Probability",fontsize=11)
        ax.set_ylabel("Observed Fraction of Positives",fontsize=11)
        ax.set_title(lbl,fontweight="bold",fontsize=11)
        ax.legend(fontsize=7.5); ax.grid(alpha=0.25)
        ax.set_xlim(0,1); ax.set_ylim(0,1)
    plt.tight_layout()
    save_fig(fig,"Fig9_Calibration")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 10: Confusion Matrices (2×2 grid — 2 thresholds each scenario)
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(2,2,figsize=(14,11))
    fig.suptitle("Figure 10: Confusion Matrices at Youden & 0.5 Thresholds",
                 fontsize=13,fontweight="bold")
    for row_i,(yt,yp,opt,lbl,bn) in enumerate([
        (yaA,ypA,optA,f"Scenario A ({bnA})",bnA),
        (yaB,ypB,optB,f"Scenario B ({bnB})",bnB)]):
        for col_i,(thr,thr_lbl) in enumerate([(opt,"Youden"),(0.5,"Default=0.5")]):
            ax = axes[row_i,col_i]
            cm = confusion_matrix(yt,(yp>=thr).astype(int))
            disp = ConfusionMatrixDisplay(cm,display_labels=["No KD","KD"])
            disp.plot(ax=ax,colorbar=False,cmap="Blues")
            tn_,fp_,fn_,tp_ = cm.ravel()
            sens = tp_/(tp_+fn_) if (tp_+fn_)>0 else 0
            spec = tn_/(tn_+fp_) if (tn_+fp_)>0 else 0
            ppv  = tp_/(tp_+fp_) if (tp_+fp_)>0 else 0
            npv  = tn_/(tn_+fn_) if (tn_+fn_)>0 else 0
            ax.set_title(
                f"{lbl} | {thr_lbl} (thr={thr:.3f})\n"
                f"Sens={sens:.3f}  Spec={spec:.3f}  PPV={ppv:.3f}  NPV={npv:.3f}",
                fontsize=9,fontweight="bold")
    plt.tight_layout()
    save_fig(fig,"Fig10_ConfusionMatrices")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 11: Threshold Analysis
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(1,2,figsize=(16,7))
    fig.suptitle("Figure 11: Clinical Threshold Analysis — Sensitivity/Specificity Trade-off",
                 fontsize=13,fontweight="bold")
    for ax,(td,opt,lbl) in zip(axes,[
        (tdA,optA,"Scenario A"),(tdB,optB,"Scenario B")]):
        ax.plot(td["Threshold"],td["Sensitivity"],lw=2.2,color=BLUE,label="Sensitivity")
        ax.plot(td["Threshold"],td["Specificity"],lw=2.2,color=GREEN,label="Specificity")
        ax.plot(td["Threshold"],td["F1"],         lw=2,  color=RED,  ls="--",label="F1 Score")
        ax.plot(td["Threshold"],td["PPV"],         lw=1.8,color=PURPLE,ls=":",label="PPV (Precision)")
        ax.plot(td["Threshold"],td["NPV"],         lw=1.8,color=ORANGE,ls=":",label="NPV")
        ax.axvline(opt,color="black",ls=":",lw=2,label=f"Youden={opt:.3f}")
        ax.fill_betweenx([0,1],[opt-0.02,opt-0.02],[opt+0.02,opt+0.02],alpha=0.15,color="gold")
        ax.set_xlabel("Decision Threshold",fontsize=11)
        ax.set_ylabel("Metric Value",fontsize=11)
        ax.set_title(lbl,fontweight="bold",fontsize=11)
        ax.legend(fontsize=9); ax.grid(alpha=0.25)
        ax.set_xlim(0,1); ax.set_ylim(0,1.05)
    plt.tight_layout()
    save_fig(fig,"Fig11_ThresholdAnalysis")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 12: Decision Curve Analysis
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(1,2,figsize=(16,7))
    fig.suptitle("Figure 12: Decision Curve Analysis — Net Clinical Benefit",
                 fontsize=14,fontweight="bold")
    for ax,(yt,yp,lbl,bn) in zip(axes,[
        (yaA,ypA,"Scenario A",bnA),
        (yaB,ypB,"Scenario B",bnB)]):
        th,nbm,nba,nbn = dca(yt,yp)
        ax.plot(th,nbm,lw=2.5,color=BLUE,label=f"Best Model ({bn})")
        ax.plot(th,nba,lw=2,color=RED,ls="--",label="Treat All")
        ax.plot(th,nbn,lw=2,color="grey",ls=":",label="Treat None")
        ax.fill_between(th,[max(m,0) for m in nbm],[max(m,0) for m in nba],
                        where=[m>a for m,a in zip(nbm,nba)],
                        alpha=0.12,color=BLUE,label="Net benefit zone")
        ax.set_xlim(0,0.8); ax.set_ylim(-0.05,max(max(nbm),0.01)+0.05)
        ax.set_xlabel("Threshold Probability",fontsize=11)
        ax.set_ylabel("Net Benefit",fontsize=11)
        ax.set_title(lbl,fontweight="bold",fontsize=11)
        ax.legend(fontsize=9); ax.grid(alpha=0.25)
        ax.axhline(0,color="black",lw=0.8)
    plt.tight_layout()
    save_fig(fig,"Fig12_DCA")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # TABLE 6 — DeLong Pairwise (both scenarios)
    # ─────────────────────────────────────────────────────────────────────────
    for suf,lbl,dl_df in [("ScenA","A",dlA),("ScenB","B",dlB)]:
        add_table_page(pdf, dl_df,
            f"Table 6: DeLong Pairwise AUC Comparison — Scenario {lbl} (*p<0.05)",
            fontsize=8)

    # PAGE — FIGURE 13: DeLong Heatmap
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(1,2,figsize=(16,7))
    fig.suptitle("Figure 13: DeLong Pairwise AUC Comparison — P-value Matrix",
                 fontsize=13,fontweight="bold")
    for ax,(dl_df,lbl) in zip(axes,[(dlA,"Scenario A"),(dlB,"Scenario B")]):
        mn_list = model_names   # ← FIX: was `list(models.keys()) if hasattr(models,"keys") else model_names`
        mat = np.ones((len(mn_list),len(mn_list)))
        for _,row_dl in dl_df.iterrows():
            i = mn_list.index(row_dl["Model A"]) if row_dl["Model A"] in mn_list else -1
            j = mn_list.index(row_dl["Model B"]) if row_dl["Model B"] in mn_list else -1
            if i>=0 and j>=0:
                mat[i,j] = row_dl["P_value"]
                mat[j,i] = row_dl["P_value"]
        sns.heatmap(mat, annot=True, fmt=".3f", cmap="RdYlGn_r", vmin=0, vmax=0.1,
                    xticklabels=[m[:10] for m in mn_list],
                    yticklabels=[m[:10] for m in mn_list],
                    ax=ax, linewidths=0.5, cbar_kws={"label":"p-value"})
        ax.set_title(f"{lbl}\n(Green = significant difference)",fontweight="bold")
    plt.tight_layout()
    save_fig(fig,"Fig13_DeLong_Heatmap")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 14: SHAP Beeswarm (Summary)
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(1,2,figsize=(18,9))
    fig.suptitle("Figure 14: SHAP Beeswarm — Feature Impact Direction on KD Prediction",
                 fontsize=13,fontweight="bold")
    for ax,(sv,Xsh,ft,lbl,bn) in zip(axes,[
        (svA,XshA,ftA,f"Scenario A ({bnA})",bnA),
        (svB,XshB,ftB,f"Scenario B ({bnB})",bnB)]):
        plt.sca(ax)
        try:
            shap.summary_plot(sv, Xsh, feature_names=ft[:sv.shape[1]],
                              plot_type="dot", show=False, max_display=15, alpha=0.6)
        except Exception as e:
            ax.text(0.5,0.5,f"SHAP plot error:\n{str(e)}",ha="center",va="center",
                    transform=ax.transAxes)
        ax.set_title(f"{lbl}",fontsize=10,fontweight="bold")
    plt.tight_layout()
    save_fig(fig,"Fig14_SHAP_Beeswarm")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 15: SHAP Bar Chart (Global Importance)
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(1,2,figsize=(16,8))
    fig.suptitle("Figure 15: SHAP Mean |SHAP| — Global Feature Importance",
                 fontsize=13,fontweight="bold")
    for ax,(sd,lbl) in zip(axes,[(sdA,"Scenario A"),(sdB,"Scenario B")]):
        top = sd.head(15)
        vals = top["Mean_abs_SHAP"].values[::-1]
        feats = top["Feature"].values[::-1]
        cmap_shap = plt.cm.RdBu_r
        colors_shap = [cmap_shap(v/max(vals)) for v in vals]
        bars = ax.barh(range(len(top)),vals,color=colors_shap,edgecolor="black",lw=0.5)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(feats,fontsize=10)
        ax.set_xlabel("Mean |SHAP Value| (impact on prediction)",fontsize=10)
        ax.set_title(lbl,fontweight="bold",fontsize=11)
        ax.grid(axis="x",alpha=0.3)
        for bar,v in zip(bars,vals):
            ax.text(bar.get_width()+0.0005,bar.get_y()+bar.get_height()/2,
                    f"{v:.4f}",va="center",fontsize=8)
    plt.tight_layout()
    save_fig(fig,"Fig15_SHAP_Bar")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # TABLE 7 — SHAP Top-10 comparison
    # ─────────────────────────────────────────────────────────────────────────
    top10A = sdA.head(10).reset_index(drop=True)
    top10B = sdB.head(10).reset_index(drop=True)
    merged = pd.DataFrame({
        "Rank":range(1,11),
        "Scen A Feature":top10A["Feature"],
        "A Mean|SHAP|":top10A["Mean_abs_SHAP"].round(5),
        "Scen B Feature":top10B["Feature"],
        "B Mean|SHAP|":top10B["Mean_abs_SHAP"].round(5)
    })
    add_table_page(pdf, merged, "Table 7: SHAP Top-10 Features — Scenario A vs B", fontsize=9)

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 16: SHAP Dependence Plots (3 per scenario)
    # ─────────────────────────────────────────────────────────────────────────
    for scen_i,(sv,Xsh,ft,lbl,bn) in enumerate([
        (svA,XshA,ftA,"Scenario A",bnA),
        (svB,XshB,ftB,"Scenario B",bnB)]):
        top_feats_dep = ft[:min(3,len(ft))]
        fig,axes = plt.subplots(1,3,figsize=(18,5))
        fig.suptitle(f"Figure {16+scen_i}: SHAP Dependence Plots — {lbl} ({bn})",
                     fontsize=13,fontweight="bold")
        for ax,feat in zip(axes,top_feats_dep):
            if feat not in ft: ax.axis("off"); continue
            plt.sca(ax)
            try:
                shap.dependence_plot(feat, sv, Xsh,
                    feature_names=ft[:sv.shape[1]], ax=ax, show=False, alpha=0.6)
                ax.set_title(f"SHAP Dependence: {feat}",fontsize=11,fontweight="bold")
            except Exception as e:
                ax.text(0.5,0.5,f"Error:\n{feat}",ha="center",va="center",transform=ax.transAxes)
        plt.tight_layout()
        save_fig(fig,f"Fig{16+scen_i}_SHAP_Dependence_{lbl.replace(' ','_')}")
        pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 18: SHAP Waterfall for 3 example patients
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(1,3,figsize=(18,7))
    fig.suptitle("Figure 18: SHAP Waterfall — Individual Patient Explanations (Scenario A)",
                 fontsize=13,fontweight="bold")
    try:
        mdlA_type = type(mdlA).__name__
        is_tree_A = (hasattr(mdlA,"estimators_") or
                     "XGB" in mdlA_type or "LGBM" in mdlA_type or
                     "Gradient" in mdlA_type)
        if is_tree_A:
            expl_w = shap.TreeExplainer(mdlA)
        else:
            # Surrogate RF: instant alternative for SVM/MLP
            print(f"  INFO Waterfall: RF surrogate for {mdlA_type}")
            surr_w = RandomForestClassifier(n_estimators=100,max_depth=5,
                                            random_state=RS,n_jobs=-1)
            surr_w.fit(XsA,(ypA>=optA).astype(int))
            expl_w = shap.TreeExplainer(surr_w)
        sv_w      = expl_w(XsA[:100])
        kd_idx    = np.where(yaA[:100]==1)[0]
        nokd_idx  = np.where(yaA[:100]==0)[0]
        probs_100 = ypA[:100]
        border_idx = np.argsort(np.abs(probs_100-optA))
        sample_indices = [
            kd_idx[0]   if len(kd_idx)>0   else 0,
            nokd_idx[0] if len(nokd_idx)>0 else 1,
            border_idx[0]
        ]
        titles = ["KD Patient (Predicted Positive)",
                  "No-KD Patient (Predicted Negative)",
                  "Borderline Patient (Near Threshold)"]
        for ax,idx,title in zip(axes,sample_indices,titles):
            plt.sca(ax)
            try:
                shap.plots.waterfall(sv_w[idx],max_display=10,show=False)
                ax.set_title(title,fontsize=9,fontweight="bold")
            except Exception as we:
                ax.text(0.5,0.5,f"Patient #{idx}\np={probs_100[idx]:.3f}",
                        ha="center",va="center",transform=ax.transAxes,fontsize=10)
                ax.axis("off")
    except Exception as e:
        for ax,txt in zip(axes,["KD Patient","No-KD Patient","Borderline"]):
            ax.text(0.5,0.5,f"{txt}\n(error: {str(e)[:60]})",
                    ha="center",va="center",transform=ax.transAxes,fontsize=10)
            ax.axis("off")
    plt.tight_layout()
    save_fig(fig,"Fig18_SHAP_Waterfall")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 19: Permutation Importance
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(1,2,figsize=(16,8))
    fig.suptitle("Figure 19: Permutation Importance (AUC Decrease) — Robustness Check",
                 fontsize=13,fontweight="bold")
    for ax,(pm,lbl) in zip(axes,[(pmA,"Scenario A"),(pmB,"Scenario B")]):
        top = pm.head(15)
        vals = top["Mean_AUC_decrease"].values[::-1]
        feats = top["Feature"].values[::-1]
        errs = top["SD"].values[::-1]
        colors_p = [RED if v>0 else "grey" for v in vals]
        ax.barh(range(len(top)),vals,xerr=errs,color=colors_p,edgecolor="black",
                lw=0.5,capsize=3,alpha=0.85)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(feats,fontsize=9)
        ax.axvline(0,color="grey",ls="--",lw=0.8)
        ax.set_xlabel("Mean AUC Decrease (±1 SD)",fontsize=11)
        ax.set_title(lbl,fontweight="bold",fontsize=11)
        ax.grid(axis="x",alpha=0.3)
        ax.text(0.98,0.02,"Red = important feature",transform=ax.transAxes,
                ha="right",fontsize=8,color="grey")
    plt.tight_layout()
    save_fig(fig,"Fig19_Permutation")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # TABLE for permutation
    perm_merged = pd.merge(
        pmA.head(15)[["Feature","Mean_AUC_decrease","SD"]].rename(
            columns={"Mean_AUC_decrease":"A_Mean","SD":"A_SD"}),
        pmB.head(15)[["Feature","Mean_AUC_decrease","SD"]].rename(
            columns={"Mean_AUC_decrease":"B_Mean","SD":"B_SD"}),
        on="Feature", how="outer"
    ).round(5).fillna("N/A")
    add_table_page(pdf, perm_merged, "Table 8: Permutation Importance — Scenario A vs B", fontsize=9)

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 20: Temporal Validation Bar Chart
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(1,2,figsize=(16,7))
    fig.suptitle("Figure 20: Temporal Validation — OOF vs Holdout AUC Comparison",
                 fontsize=13,fontweight="bold")
    for ax,(prf,lbl) in zip(axes,[(pA,"Scenario A"),(pB,"Scenario B")]):
        m_names_t  = prf["Model"].tolist()
        oof_aucs_t = prf["AUC"].tolist()
        temp_aucs_t = []
        for m in m_names_t:
            v = prf[prf["Model"]==m]["Temporal_AUC"].values[0]
            temp_aucs_t.append(float(v) if v != "N/A" and str(v) != "nan" else np.nan)
        x_t = np.arange(len(m_names_t)); w_t = 0.35
        b1 = ax.bar(x_t-w_t/2, oof_aucs_t, w_t, color=BLUE, alpha=0.85,
                    label=f"{N_SPLITS}-Fold OOF AUC", edgecolor="black", lw=0.6)
        b2 = ax.bar(x_t+w_t/2, temp_aucs_t, w_t, color=ORANGE, alpha=0.85,
                    label="Temporal Holdout AUC", edgecolor="black", lw=0.6)
        for b,v in zip(list(b1)+list(b2), oof_aucs_t+temp_aucs_t):
            if not np.isnan(v) if isinstance(v,float) else True:
                ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
                        f"{v:.3f}", ha="center", va="bottom", fontsize=7.5, fontweight="bold")
        ax.set_xticks(x_t)
        ax.set_xticklabels([m.replace(" ","\n") for m in m_names_t],fontsize=8)
        ax.set_ylabel("AUC",fontsize=11); ax.set_ylim(0.4,1.05)
        ax.axhline(0.8,color=GREEN,ls="--",lw=1.2,alpha=0.7,label="AUC=0.8")
        ax.set_title(lbl,fontweight="bold",fontsize=11)
        ax.legend(fontsize=9); ax.grid(axis="y",alpha=0.3)
    plt.tight_layout()
    save_fig(fig,"Fig20_TemporalValidation")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 21: Scenario A vs B Head-to-Head
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(2,2,figsize=(16,12))
    fig.suptitle("Figure 21: Scenario A vs B — Comprehensive Head-to-Head Comparison",
                 fontsize=13,fontweight="bold")
    x_pos2 = np.arange(len(model_names)); w2 = 0.35

    for (ax,met,ylabel,ymin,ymax), let in zip([
        (axes[0,0],"AUC","AUC",0.4,1.05),
        (axes[0,1],"AUPRC","AUPRC",0.0,1.05),
        (axes[1,0],"Brier","Brier Score (↓ better)",0.0,0.4),
        (axes[1,1],"MCC","Matthews Corr Coef",-0.2,1.0),
    ], "ABCD"):
        vA = [float(pA[pA["Model"]==m][met].values[0]) if m in pA["Model"].values else 0 for m in model_names]
        vB = [float(pB[pB["Model"]==m][met].values[0]) if m in pB["Model"].values else 0 for m in model_names]
        ax.bar(x_pos2-w2/2,vA,w2,color=BLUE,alpha=0.85,label="Scenario A",edgecolor="black",lw=0.6)
        ax.bar(x_pos2+w2/2,vB,w2,color=ORANGE,alpha=0.85,label="Scenario B",edgecolor="black",lw=0.6)
        ax.set_xticks(x_pos2)
        ax.set_xticklabels([m.replace(" ","\n") for m in model_names],fontsize=7.5)
        ax.set_ylabel(ylabel,fontsize=10); ax.set_ylim(ymin,ymax)
        ax.set_title(f"({let}) {met}",fontweight="bold",fontsize=11)
        ax.legend(fontsize=9); ax.grid(axis="y",alpha=0.3)
        if met=="AUC":
            ax.axhline(0.8,color=GREEN,ls="--",lw=1.2,alpha=0.7)
    plt.tight_layout()
    save_fig(fig,"Fig21_ScenarioComparison")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 22: Baseline vs ML (ROC + Bar)
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(1,2,figsize=(16,7))
    fig.suptitle("Figure 22: Creatinine-Only Baseline vs Best ML Models",
                 fontsize=13,fontweight="bold")
    ax = axes[0]
    names_comp = ["Baseline\n(Creatinine-LR)",f"Scenario A\n{bnA}",f"Scenario B\n{bnB}"]
    aucs_comp  = [auc_bl, auc_A_best, auc_B_best]
    colors_comp = [PURPLE, BLUE, ORANGE]
    b_comp = ax.bar(names_comp,aucs_comp,color=colors_comp,edgecolor="black",width=0.5,alpha=0.85)
    for bar,v in zip(b_comp,aucs_comp):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.005,
                f"{v:.4f}",ha="center",fontsize=12,fontweight="bold")
    ax.axhline(0.8,color=GREEN,ls="--",lw=1.5,label="AUC=0.8")
    ax.set_ylim(0.4,1.05); ax.set_ylabel("AUC (Bootstrap 95%CI)",fontsize=11)
    ax.set_title("(A) AUC vs Creatinine Baseline",fontweight="bold")
    ax.grid(axis="y",alpha=0.3); ax.legend(fontsize=10)
    ax.text(0,auc_bl+0.01,f"95%CI: {lo_bl:.3f}–{hi_bl:.3f}",fontsize=8,color=PURPLE)

    ax2 = axes[1]
    fpr_bl,tpr_bl,_ = roc_curve(oof_tbl,oof_bl)
    fpr_a, tpr_a, _ = roc_curve(otA,oofA[bnA])
    fpr_b, tpr_b, _ = roc_curve(otB,oofB[bnB])
    ax2.plot([0,1],[0,1],"k--",lw=1.2,alpha=0.5)
    ax2.plot(fpr_bl,tpr_bl,color=PURPLE,lw=2.5,label=f"Baseline LR (AUC={auc_bl:.3f})")
    ax2.plot(fpr_a, tpr_a, color=BLUE,  lw=2.5,label=f"Scen A {bnA} (AUC={auc_A_best:.3f})")
    ax2.plot(fpr_b, tpr_b, color=ORANGE,lw=2.5,label=f"Scen B {bnB} (AUC={auc_B_best:.3f})")
    ax2.fill_between(fpr_a,tpr_bl_interp := np.interp(fpr_a,fpr_bl,tpr_bl),tpr_a,
                     alpha=0.1,color=BLUE,label="Improvement A>Baseline")
    ax2.set_xlabel("False Positive Rate",fontsize=11); ax2.set_ylabel("True Positive Rate",fontsize=11)
    ax2.set_title("(B) ROC Comparison",fontweight="bold")
    ax2.legend(fontsize=9); ax2.grid(alpha=0.25)
    plt.tight_layout()
    save_fig(fig,"Fig22_BaselineComparison")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 23: Per-Fold AUC Stability (Box plots)
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(1,2,figsize=(16,7))
    fig.suptitle("Figure 23: Per-Fold AUC Stability — Cross-Validation Variance",
                 fontsize=13,fontweight="bold")
    for ax,(fold_a_dict,lbl) in zip(axes,[(faA,"Scenario A"),(faB,"Scenario B")]):
        data_box = [fold_a_dict[m] for m in model_names if m in fold_a_dict and len(fold_a_dict[m])>0]
        labels_box = [m for m in model_names if m in fold_a_dict and len(fold_a_dict[m])>0]
        if data_box:
            bp = ax.boxplot(data_box,patch_artist=True,notch=False,
                            medianprops=dict(color="black",linewidth=2))
            for patch,col in zip(bp["boxes"],PAL7):
                patch.set_facecolor(col); patch.set_alpha(0.75)
            ax.set_xticks(range(1,len(labels_box)+1))
            ax.set_xticklabels([l.replace(" ","\n") for l in labels_box],fontsize=8)
        ax.axhline(0.8,color=GREEN,ls="--",lw=1.2,alpha=0.7,label="AUC=0.8")
        ax.set_ylabel("AUC per Fold",fontsize=11)
        ax.set_title(lbl,fontweight="bold",fontsize=11)
        ax.legend(fontsize=9); ax.grid(axis="y",alpha=0.3)
        ax.set_ylim(0.4,1.05)
    plt.tight_layout()
    save_fig(fig,"Fig23_CVStability")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 24: Score Distribution (Predicted Probability by KD)
    # ─────────────────────────────────────────────────────────────────────────
    fig,axes = plt.subplots(1,2,figsize=(16,7))
    fig.suptitle("Figure 24: Predicted Probability Distribution by True KD Status",
                 fontsize=13,fontweight="bold")
    for ax,(yt,yp,opt,lbl,bn) in zip(axes,[
        (yaA,ypA,optA,f"Scenario A ({bnA})",bnA),
        (yaB,ypB,optB,f"Scenario B ({bnB})",bnB)]):
        ax.hist(yp[yt==0],bins=40,alpha=0.6,color=GREEN,label="No KD (True Neg)",
                edgecolor="white",density=True)
        ax.hist(yp[yt==1],bins=40,alpha=0.6,color=RED,label="KD (True Pos)",
                edgecolor="white",density=True)
        ax.axvline(opt,color="black",ls=":",lw=2.5,label=f"Youden={opt:.3f}")
        ax.axvline(0.5,color="grey",ls="--",lw=1.5,alpha=0.6,label="Default=0.5")
        ax.fill_betweenx([0,ax.get_ylim()[1] if ax.get_ylim()[1]>0 else 5],
                         [opt,opt],[0.5,0.5],alpha=0.08,color="gold")
        ax.set_xlabel("Predicted Probability of KD",fontsize=11)
        ax.set_ylabel("Density",fontsize=11)
        ax.set_title(lbl,fontweight="bold",fontsize=11)
        ax.legend(fontsize=9); ax.grid(alpha=0.25)
    plt.tight_layout()
    save_fig(fig,"Fig24_ScoreDistribution")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # TABLE 9 — Threshold Table (key thresholds)
    # ─────────────────────────────────────────────────────────────────────────
    # Show subset around Youden threshold for readability
    thr_select_A = tdA[(tdA["Threshold"]>=0.1)&(tdA["Threshold"]<=0.9)].iloc[::4].reset_index(drop=True)
    add_table_page(pdf, thr_select_A, "Table 9: Threshold Analysis Table — Scenario A", fontsize=8)
    thr_select_B = tdB[(tdB["Threshold"]>=0.1)&(tdB["Threshold"]<=0.9)].iloc[::4].reset_index(drop=True)
    add_table_page(pdf, thr_select_B, "Table 10: Threshold Analysis Table — Scenario B", fontsize=8)

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 25: Radar Chart — Multi-metric Comparison (Best Models)
    # ─────────────────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(16,8))
    fig.suptitle("Figure 25: Multi-Metric Radar Chart — Best Models vs Baseline",
                 fontsize=13,fontweight="bold")
    cats = ["AUC","AUPRC","1-Brier","MCC","BalAcc","F1"]
    N_cat = len(cats)
    angles = np.linspace(0,2*np.pi,N_cat,endpoint=False).tolist()
    angles += angles[:1]

    def get_radar_vals(prf, best_nm):
        row = prf[prf["Model"]==best_nm].iloc[0]
        auc_   = float(row["AUC"])
        auprc_ = float(row["AUPRC"])
        brier_ = 1 - float(row["Brier"])
        mcc_   = float(row["MCC"])
        bal_   = float(row["BalAcc"])
        f1_    = float(row["F1"])
        return [auc_,auprc_,brier_,mcc_,bal_,f1_]

    vA_r = get_radar_vals(pA,bnA)
    vB_r = get_radar_vals(pB,bnB)
    # Baseline approximate values
    vBL_r = [auc_bl,
              average_precision_score(oof_tbl,oof_bl),
              1-brier_score_loss(oof_tbl,oof_bl),
              matthews_corrcoef(oof_tbl,(oof_bl>0.5).astype(int)),
              balanced_accuracy_score(oof_tbl,(oof_bl>0.5).astype(int)),
              f1_score(oof_tbl,(oof_bl>0.5).astype(int))]

    ax_r = fig.add_subplot(111,polar=True)
    for vals,label,color in [(vA_r,f"Scen A ({bnA})",BLUE),
                              (vB_r,f"Scen B ({bnB})",ORANGE),
                              (vBL_r,"Baseline (Creatinine-LR)",PURPLE)]:
        v_plot = vals+vals[:1]
        ax_r.plot(angles,v_plot,lw=2.5,color=color,label=label)
        ax_r.fill(angles,v_plot,alpha=0.12,color=color)
    ax_r.set_xticks(angles[:-1])
    ax_r.set_xticklabels(cats,fontsize=12,fontweight="bold")
    ax_r.set_ylim(0,1)
    ax_r.set_yticks([0.2,0.4,0.6,0.8,1.0])
    ax_r.grid(alpha=0.35)
    ax_r.legend(loc="upper right",bbox_to_anchor=(1.35,1.15),fontsize=10)
    ax_r.set_title("Higher = Better (all axes normalized 0–1)\n1-Brier shown for Brier score",
                    fontsize=9,pad=25,style="italic")
    plt.tight_layout()
    save_fig(fig,"Fig25_RadarChart")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()

    # ─────────────────────────────────────────────────────────────────────────
    # PAGE — FIGURE 26: Feature Distribution Violin (Derived Features)
    # ─────────────────────────────────────────────────────────────────────────
    derived_feats = [c for c in ["NLR","PLR","MLR","ELR","SII","AISI","HI",
                                  "K_NA_RATIO","ANEMIA_SCORE"] if c in df.columns][:9]
    fig,axes = plt.subplots(3,3,figsize=(16,12))
    fig.suptitle("Figure 26: Derived Inflammatory Index Distribution by KD Status",
                 fontsize=13,fontweight="bold")
    axes = axes.flatten()
    for ax,col in zip(axes,derived_feats):
        data_v2 = [g0[col].dropna().clip(upper=g0[col].quantile(0.95)).values,
                   g1[col].dropna().clip(upper=g1[col].quantile(0.95)).values]
        parts = ax.violinplot(data_v2,positions=[0,1],showmedians=True,showextrema=True)
        for pc,color in zip(parts["bodies"],[GREEN,RED]):
            pc.set_facecolor(color); pc.set_alpha(0.7)
        parts["cmedians"].set_colors("black")
        ax.set_xticks([0,1]); ax.set_xticklabels(["No KD","KD"],fontsize=9)
        ax.set_title(col,fontweight="bold",fontsize=11)
        ax.grid(axis="y",alpha=0.3)
    for ax in axes[len(derived_feats):]: ax.axis("off")
    plt.tight_layout()
    save_fig(fig,"Fig26_DerivedFeatureViolins")
    pdf.savefig(fig,bbox_inches="tight"); plt.close()



# ═══════════════════════════════════════════════════════════════════════════════
# 11. AUTO-DOWNLOAD (Google Colab)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[Step 7/9] Triggering downloads ...")
try:
    from google.colab import files
    import zipfile

    with zipfile.ZipFile("/content/DFU_Results.zip", "w") as zf:
        for f in os.listdir("/content"):
            if f.endswith(".pdf"):
                zf.write(f"/content/{f}", f)
        for f in os.listdir(MDL_DIR):
            zf.write(f"{MDL_DIR}/{f}", f"models/{f}")
        for f in os.listdir(FIG_DIR):
            zf.write(f"{FIG_DIR}/{f}", f"figures/{f}")
        for f in os.listdir(RES_DIR):
            zf.write(f"{RES_DIR}/{f}", f"results/{f}")

    print("✓ ZIP created → /content/DFU_Results.zip")
    files.download("/content/DFU_Results.zip")
    print("✓ Download triggered")
except ImportError:
    print("ℹ  Not in Colab — all files saved locally.")
    print(f"  Main PDF: {OUT_PDF}")


[Step 5/9] Generating ALL figures and building PDF ...
  INFO Waterfall: RF surrogate for LogisticRegression

[Step 7/9] Triggering downloads ...
✓ ZIP created → /content/DFU_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download triggered
